# Week 5 사전 점검 — 로우레벨 검증 (Low-level Check)

본격적인 retrieval 실험(5주차)에 들어가기 전에, **RAG 파이프라인의 밑단 부품이 우리 데이터에서 실제로 잘 작동하는지 육안·수치로 확인**한다. 

점검 대상 :
- **A. 청킹**: 확정된 G2 설정으로 실제 잘린 chunk가 의미 단위를 지키는가 (한글·영어 각 1개 문서)
- **B. 임베딩**: e5-base가 유방암 도메인 문장의 의미를 제대로 구분하는가

> 정석은 이 로우레벨 확인을 실험 **전에** 하는 것이나, 이번엔 시간 제약상 end-to-end 실험을 먼저 하고 여기서 **사후 검증**한다. (그 순서 뒤집힘도 기록으로 남긴다.)

---
## A. 청킹 육안 확인

확정된 G2(한 540/80, 영 620/90)로 두 문서를 잘라, chunk가 **문장 중간에서 끊기지 않는지 / 표가 깨지지 않는지 / 의미 단위가 유지되는지**를 직접 본다.

In [1]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

PROJECT_ROOT = Path.cwd().parent
PDF_ROOT = PROJECT_ROOT / "data" / "raw" / "pdf"

# 한글 1개 + 영어 1개 (한글: 표 많은 대한유방암학회 / 영어: 골든셋 출제한 NCCN)
KO_PDF = PDF_ROOT / "kbcs" / "kbcs_korean_breast_cancer_guideline_2023.pdf"
EN_PDF = PDF_ROOT / "nccn" / "nccn_breast_cancer_screening_diagnosis_patient.pdf"

# 확정된 G2 크기
SIZE = {"ko": 540, "en": 620}
OVERLAP = {"ko": 80, "en": 90}
SEPARATORS = ["\n\n", "\n", ". ", " ", ""]

def load_and_split(pdf_path, lang):
    docs = PyMuPDFLoader(str(pdf_path)).load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=SIZE[lang], chunk_overlap=OVERLAP[lang],
        separators=SEPARATORS, length_function=len)
    return splitter.split_documents(docs)

ko_chunks = load_and_split(KO_PDF, "ko")
en_chunks = load_and_split(EN_PDF, "en")
print(f"[KO] {KO_PDF.name}: 페이지 {len(PyMuPDFLoader(str(KO_PDF)).load())} -> chunk {len(ko_chunks)}")
print(f"[EN] {EN_PDF.name}: 페이지 {len(PyMuPDFLoader(str(EN_PDF)).load())} -> chunk {len(en_chunks)}")

[KO] kbcs_korean_breast_cancer_guideline_2023.pdf: 페이지 270 -> chunk 1115
[EN] nccn_breast_cancer_screening_diagnosis_patient.pdf: 페이지 52 -> chunk 176


In [2]:
import random
random.seed(42)

def show_chunks(chunks, label, n=4):
    print("=" * 70)
    print(f"### {label} — 랜덤 {n}개 chunk 육안 확인")
    print("=" * 70)
    for idx in random.sample(range(len(chunks)), min(n, len(chunks))):
        c = chunks[idx]
        text = c.page_content
        page = c.metadata.get("page", "?")
        # 끝이 문장부호로 안 끝나면 '중간에서 잘림' 의심 표시
        cut = "" if text.rstrip().endswith((".", "。", "다.", "요.", "!", "?", ":", ")")) else "  <-- 문장 중간에서 끊겼을 수 있음"
        print(f"\n--- chunk #{idx} (p.{page}, {len(text)}자){cut}")
        print(text)
        print("-" * 40)

show_chunks(ko_chunks, "한글 (kbcs)", n=4)

### 한글 (kbcs) — 랜덤 4개 chunk 육안 확인

--- chunk #228 (p.75, 67자)  <-- 문장 중간에서 끊겼을 수 있음
의 병합요법이 병리학적 완전관해율을 높이고 무사고생존기간 (EFS)을 개선하
기 위해 추천된다
2
B
[216, 217]
----------------------------------------

--- chunk #51 (p.23, 133자)  <-- 문장 중간에서 끊겼을 수 있음
Prognostic Stage Group Table.
**	 If Oncotype Dx® is not performed, not available, or if the Oncotype Dx® score is 11 or greater for
----------------------------------------

--- chunk #563 (p.139, 339자)
pregnant breast cancer patients and outcomes of children exposed to chemotherapy in utero. 
Cancer. 2006;107(6):1219-26.
414.	 Doll DC, Ringenberg QS, Yarbro JW. Antineoplastic agents and pregnancy. Semin Oncol. 
1989;16(5):337-46.
415.	 Ebert U, Loffler H, Kirch W. Cytotoxic therapy and pregnancy. Pharmacol Ther. 1997;74(2):207-
20.
----------------------------------------

--- chunk #501 (p.129, 478자)
67 analysis from the monarchE study. Annals of Oncology. 2021;32(12):1571-81.
303.	 Olivotto IA, Bajdik CD, Ravdin PM, Speers CH, Coldman A

In [3]:
show_chunks(en_chunks, "영어 (NCCN)", n=4)

### 영어 (NCCN) — 랜덤 4개 chunk 육안 확인

--- chunk #57 (p.17, 554자)
fibroglandular breast tissue. But there are 
some areas of fatty tissue.
	
h Category D: Extremely dense 
means nearly all of the breast tissue is 
fibroglandular breast tissue. There is very 
little fatty tissue.
Non-dense breasts are defined as:
	
h Almost entirely fatty
	
h Scattered areas of fibroglandular density
Dense breasts are defined as:
	
h Heterogeneously dense
	
h Extremely dense
Those with dense breasts might benefit 
from more screening tests, in addition to a 
screening mammogram. Ask your health care 
provider for more information.
----------------------------------------

--- chunk #35 (p.12, 603자)  <-- 문장 중간에서 끊겼을 수 있음
A mammogram is a picture of the inside 
of your breast made using x-rays. During 
a mammogram, the breast is pressed 
between two plates while you stand in 
different positions. Multiple x-rays will be 
taken. A computer combines the x-rays to 
make detailed pictures.
Screening mammogram
•	


---
## B. 임베딩 도메인 유사도 확인

e5-base가 **유방암 도메인 문장**의 의미를 제대로 구분하는지 본다. 일반 문장은 잘 구분해도 의료 용어(HER2, 감시림프절 등)에선 다를 수 있으므로 도메인 문장으로 확인한다.

> **e5 접두어 규칙**: e5 계열은 문장 앞에 `query: `(질문) / `passage: `(문서)를 붙여야 성능이 제대로 난다. 아래는 대칭 유사도 비교라 모두 `query: `를 붙인다. **실제 파이프라인에서 이 접두어가 붙어 있는지도 반드시 확인할 것.**

In [4]:
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model_dir = snapshot_download(repo_id="intfloat/multilingual-e5-base", local_files_only=True)
model = SentenceTransformer(model_dir, device="cpu")
print("e5-base 로드 완료 (로컬, API 0원)")

def sim(a, b):
    # e5 대칭 유사도: 둘 다 query: 접두어
    va, vb = model.encode(["query: " + a, "query: " + b], normalize_embeddings=True)
    return float(cosine_similarity([va], [vb])[0][0])

e5-base 로드 완료 (로컬, API 0원)


In [5]:
# 유방암 도메인 문장쌍 — (기대: 관련=높음, 반대/무관=낮음)
pairs = [
    ("유방암 검진은 40-69세 무증상 여성에서 2년마다 유방촬영술로 시행한다",
     "40세부터 2년 간격으로 유방촬영 검진을 받는 것이 권장된다", "거의 동일 (높아야)"),
    ("HER2 양성 유방암은 표적 치료를 한다",
     "HER2 음성 유방암의 치료 방법", "같은 주제·반대 아형 (중간)"),
    ("감시림프절 생검은 겨드랑이 림프절 전이를 확인한다",
     "겨드랑이 림프절 절제술", "관련 시술 (중간~높음)"),
    ("유방 보존술 후 방사선 치료를 시행한다",
     "유방 전절제술", "대조되는 수술 (중간)"),
    ("유방암 검진 권고 연령",
     "사과를 매일 먹으면 건강에 좋다", "완전 무관 (낮아야)"),
]

print("=== 유방암 도메인 문장 유사도 (e5-base) ===\n")
for a, b, expect in pairs:
    s = sim(a, b)
    print(f"[{s:.4f}]  기대: {expect}")
    print(f"    A: {a}")
    print(f"    B: {b}\n")

=== 유방암 도메인 문장 유사도 (e5-base) ===

[0.9327]  기대: 거의 동일 (높아야)
    A: 유방암 검진은 40-69세 무증상 여성에서 2년마다 유방촬영술로 시행한다
    B: 40세부터 2년 간격으로 유방촬영 검진을 받는 것이 권장된다

[0.9374]  기대: 같은 주제·반대 아형 (중간)
    A: HER2 양성 유방암은 표적 치료를 한다
    B: HER2 음성 유방암의 치료 방법

[0.8939]  기대: 관련 시술 (중간~높음)
    A: 감시림프절 생검은 겨드랑이 림프절 전이를 확인한다
    B: 겨드랑이 림프절 절제술

[0.8988]  기대: 대조되는 수술 (중간)
    A: 유방 보존술 후 방사선 치료를 시행한다
    B: 유방 전절제술

[0.7923]  기대: 완전 무관 (낮아야)
    A: 유방암 검진 권고 연령
    B: 사과를 매일 먹으면 건강에 좋다



## 분석 결론

A. 청킹: 본문은 G2로 문단 단위 분할 양호. 단, 한글 문서 뒤쪽 참고문헌·색인·표 조각이 노이즈 chunk로 다수 포함됨 → 향후 references 섹션 제외 전처리 검토.

B. 임베딩: e5-base는 주제 수준 구분은 되나, ①무관 문장도 유사도 0.79로 전반적으로 높고 ②HER2 양성/음성 같은 도메인 미세 반대를 구분 못 함(0.937). → 1차 검색 변별력이 약해, 5주차 Rerank로 재정렬 보정이 필요하다는 근거.